In [1]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [2]:

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

load_dotenv()


True

In [3]:
import youtube_transcript_api
print(youtube_transcript_api.__file__)

from importlib.metadata import version
print(version("youtube-transcript-api"))

# !pip install --upgrade youtube-transcript-api


/home/vvizard/anaconda3/envs/genai/lib/python3.11/site-packages/youtube_transcript_api/__init__.py
1.2.4


In [4]:
import youtube_transcript_api
print(youtube_transcript_api.__file__)
print(dir(youtube_transcript_api))
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "wwSzpaTHyS8"

from langchain_community.document_loaders import YoutubeLoader

loader = YoutubeLoader.from_youtube_url(f"https://www.youtube.com/watch?v={video_id}", add_video_info=False)
transcript = loader.load()


/home/vvizard/anaconda3/envs/genai/lib/python3.11/site-packages/youtube_transcript_api/__init__.py
['AgeRestricted', 'CookieError', 'CookieInvalid', 'CookiePathInvalid', 'CouldNotRetrieveTranscript', 'FailedToCreateConsentCookie', 'FetchedTranscript', 'FetchedTranscriptSnippet', 'InvalidVideoId', 'IpBlocked', 'NoTranscriptFound', 'NotTranslatable', 'PoTokenRequired', 'RequestBlocked', 'Transcript', 'TranscriptList', 'TranscriptsDisabled', 'TranslationLanguageNotAvailable', 'VideoUnavailable', 'VideoUnplayable', 'YouTubeDataUnparsable', 'YouTubeRequestFailed', 'YouTubeTranscriptApi', 'YouTubeTranscriptApiException', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_api', '_errors', '_settings', '_transcripts', 'proxies']


In [7]:
def build_retriever(transcript, chunk_size=1000, chunk_overlap=200, k=4):
    # Clean text
    text = transcript[0].page_content
    text = text.replace('\xa0', ' ')
    text = ' '.join(text.split())
    
    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.create_documents([text])
    
    # Embed & store
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vector_store = FAISS.from_documents(chunks, embeddings)
    
    return vector_store.as_retriever(search_type="similarity", search_kwargs={"k": k})


def format_docs(retrieved_docs):
    return '\n\n'.join(doc.page_content for doc in retrieved_docs)


def build_chain(transcript, model="gpt-4o-mini", temperature=0.2):
    retriever = build_retriever(transcript)

    prompt = PromptTemplate(
        template="""
          You are a helpful assistant.
          Answer ONLY from the provided transcript context.
          If the context is insufficient, just say you don't know.

          {context}
          Question: {question}
        """,
        input_variables=['context', 'question']
    )

    llm = ChatOpenAI(model=model, temperature=temperature)

    parallel_chain = RunnableParallel({
        'context': retriever | RunnableLambda(format_docs),
        'question': RunnablePassthrough()
    })

    return parallel_chain | prompt | llm | StrOutputParser()

In [8]:
main_chain = build_chain(transcript)

main_chain.invoke('Can you summarize the video')

'The video explores the concept of time, discussing different perspectives on how past, present, and future exist. It presents the idea of the universe as a series of moments, akin to a movie where only the current moment is real. The video also introduces the notion of multiple "nows" due to relativity, suggesting a frozen block universe where free will might not exist. Ultimately, it proposes a growing block universe model, where time passes and the future remains open, prompting viewers to consider the reality of the past, present, and future. Additionally, it promotes a series of interactive lessons on scientific topics in collaboration with Brilliant.org.'